# 구조물 안전성 예측 — 개선 버전
**적용 기법:** EfficientNet-B5 백본 · Cross-View Attention 융합 · Label Smoothing · Cosine Annealing · 5-Fold 앙상블 · TTA

## 1. 라이브러리 및 환경 설정

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import copy
import math
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b5, EfficientNet_B5_Weights
)

from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

# ── 하이퍼파라미터 ──────────────────────────────────────────────────
CFG = {
    'IMG_SIZE'     : 456,   # EfficientNet-B5 권장 입력 크기
    'EPOCHS'       : 30,
    'LEARNING_RATE': 1e-4,
    'BATCH_SIZE'   : 16,    # B5는 메모리 소모가 크므로 16 권장 (GPU 24GB+ 환경에서는 32 가능)
    'SEED'         : 42,
    'N_FOLDS'      : 5,
    'LABEL_SMOOTH' : 0.05,  # Label Smoothing 강도
    'TTA_STEPS'    : 4,     # TTA 반복 횟수
    'PATIENCE'     : 7,     # Early Stopping patience
    'BASE_PATH'    : '/content/drive/MyDrive/MyDrive/Stability_Project',
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 데이터 압축 해제 및 로드

In [ ]:
BASE_PATH = CFG['BASE_PATH']

!mkdir -p /content/train_images /content/dev_images /content/test_images

for split, dest in [('train', '/content/train_images'),
                    ('dev',   '/content/dev_images'),
                    ('test',  '/content/test_images')]:
    zip_path = os.path.join(BASE_PATH, f'{split}.zip')
    if os.path.exists(zip_path):
        os.system(f'unzip -q "{zip_path}" -d {dest}')
        print(f'✅ {split}.zip 압축 해제 완료')
    else:
        print(f'❌ {zip_path} 없음')

In [ ]:
train_df = pd.read_csv(os.path.join(BASE_PATH, 'train.csv'))
val_df   = pd.read_csv(os.path.join(BASE_PATH, 'dev.csv'))
test_df  = pd.read_csv(os.path.join(BASE_PATH, 'sample_submission.csv'))

# 5-Fold를 위해 train + dev 합산
all_df = pd.concat([train_df, val_df], ignore_index=True)
# root_dir 컬럼 추가 (train vs dev 이미지 경로 구분)
train_df['root_dir'] = '/content/train_images/train'
val_df['root_dir']   = '/content/dev_images/dev'
all_df = pd.concat([train_df, val_df], ignore_index=True)

print(f'전체 학습 데이터: {len(all_df)}  |  테스트: {len(test_df)}')
print(all_df['label'].value_counts())

## 3. 데이터셋 및 Augmentation

In [ ]:
# ── Augmentation 정의 ─────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),   # 랜덤 패치 제거
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# TTA용 변환 목록 (추론 시 여러 버전의 이미지를 평균)
tta_transforms = [
    val_transform,  # 원본
    transforms.Compose([
        transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((int(CFG['IMG_SIZE'] * 1.1), int(CFG['IMG_SIZE'] * 1.1))),
        transforms.CenterCrop(CFG['IMG_SIZE']),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]

In [ ]:
class MultiViewDataset(Dataset):
    """
    front.png / top.png 두 장을 함께 로드하는 멀티뷰 데이터셋.
    transform_list를 넘기면 각 변환을 순서대로 적용해 TTA에도 사용 가능.
    """
    def __init__(self, df, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.is_test   = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def _load_img(self, path):
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        sample_id  = str(row['id'])
        root_dir   = row['root_dir'] if 'root_dir' in row else '/content/test_images/test'
        folder     = os.path.join(root_dir, sample_id)

        views = [
            self._load_img(os.path.join(folder, 'front.png')),
            self._load_img(os.path.join(folder, 'top.png')),
        ]

        if self.is_test:
            return views

        label = self.label_map[row['label']]
        return views, label

## 4. 모델 정의 — EfficientNet-B5 + Cross-View Attention

In [ ]:
class CrossViewAttention(nn.Module):
    """
    두 뷰의 feature vector가 서로를 참조(attend)하여 융합하는 모듈.
    - f1(front feature)이 f2(top feature)에 주의를 기울이고,
      동시에 f2가 f1에 주의를 기울임.
    - 최종 출력: [attended_1 + f1, attended_2 + f2] (residual 포함)
    """
    def __init__(self, feat_dim: int):
        super().__init__()
        self.scale = math.sqrt(feat_dim)

        # f1 → f2 방향 attention
        self.q1 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.k2 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.v2 = nn.Linear(feat_dim, feat_dim, bias=False)

        # f2 → f1 방향 attention
        self.q2 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.k1 = nn.Linear(feat_dim, feat_dim, bias=False)
        self.v1 = nn.Linear(feat_dim, feat_dim, bias=False)

        self.norm1 = nn.LayerNorm(feat_dim)
        self.norm2 = nn.LayerNorm(feat_dim)

    def forward(self, f1, f2):
        # f1이 f2를 참조
        attn_12 = torch.sigmoid((self.q1(f1) * self.k2(f2)) / self.scale)
        out1    = self.norm1(f1 + attn_12 * self.v2(f2))   # residual

        # f2가 f1을 참조
        attn_21 = torch.sigmoid((self.q2(f2) * self.k1(f1)) / self.scale)
        out2    = self.norm2(f2 + attn_21 * self.v1(f1))   # residual

        return torch.cat([out1, out2], dim=1)  # (B, feat_dim * 2)


class MultiViewEfficientNet(nn.Module):
    """
    EfficientNet-B5 백본 + CrossViewAttention 융합 분류기.
    두 뷰가 가중치를 공유(shared backbone)하여 파라미터 효율을 높임.
    """
    def __init__(self, num_classes: int = 1, dropout: float = 0.4):
        super().__init__()
        # EfficientNet-B5: output feature dim = 2048
        backbone = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        # AdaptiveAvgPool2d + Dropout + Linear(head) 제거 → feature만 추출
        self.feature_extractor = nn.Sequential(
            backbone.features,       # Conv layers
            backbone.avgpool,        # AdaptiveAvgPool2d → (B, 2048, 1, 1)
        )
        feat_dim = 2048

        self.fusion = CrossViewAttention(feat_dim)

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * 2, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes),
        )

    def extract(self, x):
        return self.feature_extractor(x).flatten(1)  # (B, 2048)

    def forward(self, views):
        f1 = self.extract(views[0])
        f2 = self.extract(views[1])
        fused = self.fusion(f1, f2)   # (B, 4096)
        return self.classifier(fused) # (B, 1)


# 간단 동작 확인
dummy_model = MultiViewEfficientNet().to(device)
dummy_views = [torch.randn(2, 3, 456, 456).to(device),
               torch.randn(2, 3, 456, 456).to(device)]
out = dummy_model(dummy_views)
print(f'Model output shape: {out.shape}')  # (2, 1)
del dummy_model, dummy_views, out

## 5. 손실 함수 및 학습 유틸

In [ ]:
class LabelSmoothingBCE(nn.Module):
    """
    Label Smoothing 적용 Binary Cross-Entropy.
    smoothing=0.05 → 실제 target을 [0.025, 0.975] 범위로 부드럽게 만들어
    모델이 과도하게 확신(0 or 1에 근접)하는 것을 방지 → Log-Loss 개선.
    """
    def __init__(self, smoothing: float = 0.05):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        # targets: 0 or 1 float tensor
        smooth_targets = targets * (1.0 - self.smoothing) + self.smoothing * 0.5
        return F.binary_cross_entropy_with_logits(logits, smooth_targets)


def compute_logloss(probs, labels):
    """대회 공식 Binary Log-Loss 계산."""
    eps = 1e-15
    p = np.clip(probs, eps, 1 - eps)
    return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0

    for views, labels in tqdm(loader, desc='Train', leave=False):
        views  = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()

        # Mixed Precision (AMP) → GPU 메모리 절약 + 속도 향상
        with torch.cuda.amp.autocast():
            logits = model(views).view(-1)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        # Gradient Clipping → 학습 안정성
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []

    for views, labels in tqdm(loader, desc='Val', leave=False):
        views  = [v.to(device) for v in views]
        with torch.cuda.amp.autocast():
            logits = model(views).view(-1)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs,  dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    logloss = compute_logloss(all_probs, all_labels)
    acc     = np.mean((all_probs > 0.5) == all_labels)
    return logloss, acc

## 6. 5-Fold 학습

In [ ]:
skf = StratifiedKFold(
    n_splits=CFG['N_FOLDS'],
    shuffle=True,
    random_state=CFG['SEED']
)

# 저장 경로
CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

fold_val_scores = []   # 각 fold 최고 val logloss 기록

for fold, (train_idx, val_idx) in enumerate(
        skf.split(all_df, all_df['label']), start=1):

    print(f'\n{"="*55}')
    print(f'  FOLD {fold} / {CFG["N_FOLDS"]}')
    print(f'{"="*55}')

    fold_train_df = all_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df   = all_df.iloc[val_idx].reset_index(drop=True)

    # Dataset & DataLoader
    train_ds = MultiViewDataset(fold_train_df, transform=train_transform)
    val_ds   = MultiViewDataset(fold_val_df,   transform=val_transform)

    train_loader = DataLoader(
        train_ds, batch_size=CFG['BATCH_SIZE'],
        shuffle=True, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG['BATCH_SIZE'],
        shuffle=False, num_workers=2, pin_memory=True
    )

    # 모델 · 손실 · 옵티마이저 초기화
    model     = MultiViewEfficientNet().to(device)
    criterion = LabelSmoothingBCE(smoothing=CFG['LABEL_SMOOTH'])

    # Differential Learning Rate:
    #   backbone은 낮은 lr, classifier/fusion은 높은 lr로 fine-tuning
    backbone_params    = list(model.feature_extractor.parameters())
    head_params        = (list(model.fusion.parameters()) +
                          list(model.classifier.parameters()))
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': CFG['LEARNING_RATE'] * 0.1},
        {'params': head_params,     'lr': CFG['LEARNING_RATE']},
    ], weight_decay=1e-4)

    # Cosine Annealing with Warmup
    total_steps   = CFG['EPOCHS'] * len(train_loader)
    warmup_steps  = 2 * len(train_loader)  # 2 epoch warmup

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.cuda.amp.GradScaler()  # Mixed Precision scaler

    best_logloss = float('inf')
    best_weights = None
    patience_cnt = 0

    for epoch in range(1, CFG['EPOCHS'] + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device
        )
        val_logloss, val_acc = validate(model, val_loader, device)

        # scheduler step은 batch 단위
        for _ in range(len(train_loader)):
            scheduler.step()

        # 현재 lr 출력
        cur_lr = optimizer.param_groups[-1]['lr']

        print(f'  Epoch [{epoch:02d}/{CFG["EPOCHS"]}] '
              f'train_loss={train_loss:.4f}  '
              f'val_logloss={val_logloss:.5f}  '
              f'val_acc={val_acc:.4f}  '
              f'lr={cur_lr:.2e}')

        # Best model 저장
        if val_logloss < best_logloss:
            best_logloss = val_logloss
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, f'{CKPT_DIR}/fold{fold}_best.pth')
            patience_cnt = 0
            print(f'  ✅ Best updated → {best_logloss:.5f}')
        else:
            patience_cnt += 1
            if patience_cnt >= CFG['PATIENCE']:
                print(f'  ⏹ Early stopping at epoch {epoch}')
                break

    fold_val_scores.append(best_logloss)
    print(f'\n  Fold {fold} best val logloss: {best_logloss:.5f}')

print(f'\n{'='*55}')
print(f'  5-Fold CV Mean Log-Loss: {np.mean(fold_val_scores):.5f}')
print(f'  Per-fold: {[f"{s:.5f}" for s in fold_val_scores]}')

## 7. TTA 추론 + 앙상블

In [ ]:
class TTADataset(Dataset):
    """단일 transform을 받아 TTA 루프에서 재사용하는 테스트 데이터셋."""
    def __init__(self, df, root_dir, transform):
        self.df        = df.reset_index(drop=True)
        self.root_dir  = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        folder    = os.path.join(self.root_dir, sample_id)
        views = [
            self.transform(Image.open(os.path.join(folder, 'front.png')).convert('RGB')),
            self.transform(Image.open(os.path.join(folder, 'top.png')).convert('RGB')),
        ]
        return views


@torch.no_grad()
def predict_tta(model, test_df, test_root, device, tta_tfms):
    """
    여러 TTA 변환을 순서대로 적용해 예측을 평균 → 단일 예측 배열 반환.
    """
    model.eval()
    all_tta_probs = []

    for tfm in tta_tfms:
        ds     = TTADataset(test_df, test_root, tfm)
        loader = DataLoader(ds, batch_size=CFG['BATCH_SIZE'],
                            shuffle=False, num_workers=2)
        probs  = []
        for views in tqdm(loader, desc='TTA inference', leave=False):
            views = [v.to(device) for v in views]
            with torch.cuda.amp.autocast():
                logits = model(views).view(-1)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_tta_probs.append(np.array(probs))

    return np.mean(all_tta_probs, axis=0)  # TTA 평균


# 5개 fold 모델 각각 TTA 추론 → 앙상블
TEST_ROOT    = '/content/test_images/test'
fold_test_probs = []

for fold in range(1, CFG['N_FOLDS'] + 1):
    print(f'\nFold {fold} TTA 추론 중...')
    model = MultiViewEfficientNet().to(device)
    model.load_state_dict(
        torch.load(f'{CKPT_DIR}/fold{fold}_best.pth', map_location=device)
    )
    probs = predict_tta(model, test_df, TEST_ROOT, device, tta_transforms)
    fold_test_probs.append(probs)
    print(f'  fold {fold} done. mean_prob={probs.mean():.4f}')

# 5-Fold 앙상블: 단순 평균 (log-odds 평균도 고려 가능)
final_probs = np.mean(fold_test_probs, axis=0)
print(f'\n앙상블 완료. final mean_prob={final_probs.mean():.4f}')

## 8. 제출 파일 생성

In [ ]:
submission = pd.DataFrame({
    'id'           : test_df['id'],
    'unstable_prob': final_probs,
    'stable_prob'  : 1.0 - final_probs,
})

submission.to_csv('submission.csv', index=False, encoding='UTF-8-sig')
print('submission.csv 저장 완료.')
print(submission.head(10))

## 부록 — 주요 변경 사항 요약

| 항목 | Baseline | 개선 버전 |
|---|---|---|
| 백본 | ResNet18 (11M) | EfficientNet-B5 (30M) |
| 이미지 크기 | 224×224 | 456×456 |
| 뷰 융합 | Concatenate | Cross-View Attention + Residual |
| 손실 함수 | BCEWithLogitsLoss | Label Smoothing BCE (α=0.05) |
| 옵티마이저 | Adam | AdamW + Differential LR |
| LR 스케줄러 | ReduceLROnPlateau | Cosine Annealing w/ Warmup |
| 학습 방식 | 단일 학습 | 5-Fold Cross Validation |
| 추론 방식 | 단일 예측 | TTA (4가지 변환) + 5-Fold 앙상블 |
| 학습 최적화 | 없음 | Mixed Precision (AMP) + Grad Clip |
| 조기 종료 | 없음 | Early Stopping (patience=7) |